# PubMed RAG: Chunking Strategy Pipeline

## Settings

In [4]:
import sys
import csv
import os
import json
import torch
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load .env
load_dotenv(find_dotenv())

# -------------------------
# Config
# -------------------------
SEED = int(os.environ.get("SEED", 42))
DATA_DIR = os.environ.get("DATA_DIR", "/workspace/data")

PARSED_DIR = os.path.join(DATA_DIR, "parsed_docs")
CHUNK_DIR = os.path.join(DATA_DIR, "chunked_docs")
train_csv_path = os.path.join(PARSED_DIR, "training_final.csv")
test_csv_path = os.path.join(PARSED_DIR, "test_final.csv")

# -------------------------
# Environment Info
# -------------------------
print(f"SEED: {SEED}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch version: {torch.__version__}")
print(f"Data dir: {DATA_DIR}")
print(f"Train CSV path: {train_csv_path}")
print(f"Test CSV path: {test_csv_path}")

SEED: 42
CUDA available: True
PyTorch version: 2.2.0
Data dir: /workspace/data
Train CSV path: /workspace/data/parsed_docs/training_final.csv
Test CSV path: /workspace/data/parsed_docs/test_final.csv


## Helpers

In [5]:
from ftfy import fix_text
import unicodedata
import html
import re
import pandas as pd

def clean_pubmed_text(text):

    if pd.isna(text):
        return ""

    # fix encoding problems
    text = fix_text(text)

    # decode html entities
    text = html.unescape(text)

    # remove html tags
    text = re.sub(r"<.*?>", " ", text)

    # normalize unicode
    text = unicodedata.normalize("NFKC", text)

    # normalize weird unicode spaces
    text = re.sub(r"[\u2000-\u200F\u202F\u205F]", " ", text)

    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

## Load Data

In [6]:
df = pd.read_csv(train_csv_path)

dup = df["pmid"].duplicated().sum()
print("Duplicate PMIDs:", dup)

df = df.drop_duplicates(
    subset=["pmid"]
).reset_index(drop=True)

print("Unique PMIDs:", df["pmid"].nunique())
print("Total rows:", len(df))


Duplicate PMIDs: 1847
Unique PMIDs: 37595
Total rows: 37595


## Chunking Experimental Setup

| Strategy          | Description                         | Typical Chunk Size |
| ----------------- | ----------------------------------- | ------------------ |
| **Section-aware** | Title + abstract sections preserved | 400–600            |
| **Fixed-length**  | Split long text uniformly           | 500                |
| **Short chunks**  | Fine-grained retrieval              | 120–200            |


In [7]:
!pip install transformers accelerate torch

### Strategy 1 — Contextual Chunking

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

llm = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=80,
    return_full_text=False
)

/opt/conda/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


KeyboardInterrupt: 

In [ ]:
def summarize_chunk(title, section, chunk):

    prompt = f"""
You are assisting a biomedical retrieval system.

Summarize the key biomedical context of the following text in ONE sentence.

Title: {title}
Section: {section}

Text:
{chunk}

Summary:
"""

    result = llm(prompt)[0]["generated_text"].strip()

    summary = result.split("Summary:")[-1].strip()

    return summary

In [ ]:
import pandas as pd
import json
import re
import time
from tqdm import tqdm
from langchain_text_splitters import RecursiveCharacterTextSplitter

start_time = time.time()

df = pd.read_csv("pubmed_articles.csv")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=[
        "\n\n",
        "\n",
        ". ",
    ]
)

chunks = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing articles"):

    pmid = row["pmid"]
    title = str(row["title"]).strip()

    # ---------- TITLE CHUNK ----------
    chunks.append({
        "pmid": pmid,
        "section": "TITLE",
        "chunk_id": f"{pmid}_TITLE",
        "chunk_text": title
    })

    sections = json.loads(row["abstract_sections"])

    for sec in sections:

        label = sec.get("label", "ABSTRACT")
        text = sec.get("text", "")

        text = re.sub(r"<.*?>", "", text).strip()

        # summarize once per section
        section_summary = summarize_chunk(title, label, text)

        # ---------- SHORT SECTION ----------
        if len(text) <= 500:

            contextual_text = (
                f"Context summary: {section_summary}\n\n"
                f"{text}"
            )

            chunks.append({
                "pmid": pmid,
                "section": label,
                "chunk_id": f"{pmid}_{label}_0",
                "chunk_text": contextual_text
            })

        # ---------- LONG SECTION ----------
        else:

            split_texts = splitter.split_text(text)

            for i, chunk in enumerate(split_texts):

                contextual_text = (
                    f"Context summary: {section_summary}\n\n"
                    f"{chunk}"
                )

                chunks.append({
                    "pmid": pmid,
                    "section": label,
                    "chunk_id": f"{pmid}_{label}_{i}",
                    "chunk_text": contextual_text
                })

fixed_df = pd.DataFrame(chunks)

fixed_df.to_csv("chunks_section_contextual.csv", index=False)

end_time = time.time()

print("\nTotal chunks:", len(fixed_df))
print("Runtime:", round(end_time - start_time, 2), "seconds")

print("\nSample output:")
print(chunk_df.head())

### Strategy 2 — Section-Aware Chunking

In [4]:
import pandas as pd
import json
import re
import html
import unicodedata
import time
from tqdm import tqdm
from pathlib import Path

from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

start_time = time.time()

# =========================
# Tokenizer (for token chunking)
# =========================

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=400,
    chunk_overlap=60,
    separators=[
        "\n\n",
        "\n",
        ". ",
    ]
)

# =========================
# Chunking
# =========================

chunks = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    pmid = str(row["pmid"])
    title = clean_pubmed_text(row.get("title", ""))

    topic_id = row.get("topic_id")
    category = row.get("category")

    # =========================
    # TITLE CHUNK
    # =========================

    if title:
        chunks.append({
            "pmid": pmid,
            "topic_id": topic_id,
            "category": category,
            "section": "TITLE",
            "chunk_id": f"{pmid}_TITLE",
            "title": title,
            "chunk_text": title
        })

    # =========================
    # ABSTRACT SECTIONS
    # =========================

    try:
        sections = json.loads(row["abstract_sections"])
    except:
        continue

    for sec in sections:

        label = sec.get("label") or "ABSTRACT"
        text = clean_pubmed_text(sec.get("text", ""))

        if not text:
            continue

        token_len = len(tokenizer(text)["input_ids"])

        # short section
        if token_len <= 400:

            chunks.append({
                "pmid": pmid,
                "topic_id": topic_id,
                "category": category,
                "section": label,
                "chunk_id": f"{pmid}_{label}_0",
                "title": title,
                "chunk_text": text
            })

        else:

            split_texts = splitter.split_text(text)

            for i, chunk in enumerate(split_texts):

                chunks.append({
                    "pmid": pmid,
                    "topic_id": topic_id,
                    "category": category,
                    "section": label,
                    "chunk_id": f"{pmid}_{label}_{i}",
                    "title": title,
                    "chunk_text": chunk
                })


# =========================
# Convert to DataFrame
# =========================

chunk_df = pd.DataFrame(chunks)

# =========================
# Deduplicate chunks
# =========================

before = len(chunk_df)

chunk_df = chunk_df.drop_duplicates(
    subset=["chunk_id"]
).reset_index(drop=True)

after = len(chunk_df)

print(f"Removed {before - after} duplicate chunks")

# =========================
# Save
# =========================

output_dir = Path(CHUNK_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "chunks_section_400token.csv"

chunk_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

end_time = time.time()

print("\nTotal chunks:", len(chunk_df))
print("Runtime:", round(end_time - start_time, 2), "seconds")

print("\nSample output:")
print(chunk_df.head())

/opt/conda/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
100% 37595/37595 [00:51<00:00, 725.87it/s]


Removed 7 duplicate chunks

Total chunks: 109234
Runtime: 54.37 seconds

Sample output:
       pmid  topic_id       category   section             chunk_id  \
0  15829955         8  Rare Diseases     TITLE       15829955_TITLE   
1  15829955         8  Rare Diseases  ABSTRACT  15829955_ABSTRACT_0   
2  15617541         8  Rare Diseases     TITLE       15617541_TITLE   
3  15617541         8  Rare Diseases  ABSTRACT  15617541_ABSTRACT_0   
4  12239580         8  Rare Diseases     TITLE       12239580_TITLE   

                                               title  \
0  A common sex-dependent mutation in a RET enhan...   
1  A common sex-dependent mutation in a RET enhan...   
2  Studying the genetics of Hirschsprung's diseas...   
3  Studying the genetics of Hirschsprung's diseas...   
4  Hirschsprung, RET-SOX and beyond: the challeng...   

                                          chunk_text  
0  A common sex-dependent mutation in a RET enhan...  
1  The identification of common varian

In [10]:
import re

def find_special_chars(text):
    return re.findall(r"[^\x00-\x7F]", str(text))

chunk_df["weird_chars"] = chunk_df["chunk_text"].apply(find_special_chars)

char_counts = (
    chunk_df["weird_chars"]
    .explode()
    .value_counts()
)

print(char_counts.head(30))

NameError: name 'chunk_df' is not defined

In [ ]:
# character length
chunk_df["char_len"] = chunk_df["chunk_text"].str.len()

# word count
chunk_df["word_count"] =chunk_df["chunk_text"].str.split().str.len()

print("Character length distribution:")
print(chunk_df["char_len"].describe())

print("\nWord count distribution:")
print(chunk_df["word_count"].describe())

### Strategy 3 — Fixed Length Chunk (>500)

In [8]:
import pandas as pd
import json
import re
import html
import unicodedata
import time
from tqdm import tqdm
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter

start_time = time.time()


# =========================
# Fixed-length splitter (500 chars)
# =========================

splitter_500 = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=[""]
)


# =========================
# Chunking
# =========================

fixed_chunks = []

for _, row in tqdm(df.iterrows(), total=len(df)):

    pmid = str(row["pmid"])
    title = clean_pubmed_text(row.get("title", ""))

    topic_id = row.get("topic_id")
    category = row.get("category")

    # =========================
    # TITLE CHUNK
    # =========================

    if title:
        fixed_chunks.append({
            "pmid": pmid,
            "topic_id": topic_id,
            "category": category,
            "section": "TITLE",
            "chunk_id": f"{pmid}_TITLE",
            "title": title,
            "chunk_text": title
        })


    # =========================
    # ABSTRACT SECTIONS
    # =========================

    try:
        sections = json.loads(row["abstract_sections"])
    except:
        continue


    # join all sections into one text
    full_text = " ".join(
        clean_pubmed_text(sec.get("text", ""))
        for sec in sections
    )


    if not full_text:
        continue


    # =========================
    # Fixed-length chunking
    # =========================

    split_texts = splitter_500.split_text(full_text)

    for i, chunk in enumerate(split_texts):

        fixed_chunks.append({
            "pmid": pmid,
            "topic_id": topic_id,
            "category": category,
            "section": "ABSTRACT",
            "chunk_id": f"{pmid}_F500_{i}",
            "title": title,
            "chunk_text": chunk
        })

# =========================
# Convert to DataFrame
# =========================

fixed_df = pd.DataFrame(fixed_chunks)

# =========================
# Deduplicate chunks
# =========================

before = len(fixed_df)

fixed_df = fixed_df.drop_duplicates(
    subset=["chunk_id"]
).reset_index(drop=True)

after = len(fixed_df)

print(f"Removed {before - after} duplicate chunks")

# =========================
# Save
# =========================

output_dir = Path(CHUNK_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "chunks_fixed500char.csv"

fixed_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

end_time = time.time()

print("\nTotal chunks:", len(fixed_df))
print("Runtime:", round(end_time - start_time, 2), "seconds")

print("\nSample output:")
print(fixed_df.head())


100% 37595/37595 [01:28<00:00, 425.30it/s]


Removed 0 duplicate chunks

Total chunks: 176626
Runtime: 90.97 seconds

Sample output:
       pmid  topic_id       category   section         chunk_id  \
0  15829955         8  Rare Diseases     TITLE   15829955_TITLE   
1  15829955         8  Rare Diseases  ABSTRACT  15829955_F500_0   
2  15829955         8  Rare Diseases  ABSTRACT  15829955_F500_1   
3  15829955         8  Rare Diseases  ABSTRACT  15829955_F500_2   
4  15617541         8  Rare Diseases     TITLE   15617541_TITLE   

                                               title  \
0  A common sex-dependent mutation in a RET enhan...   
1  A common sex-dependent mutation in a RET enhan...   
2  A common sex-dependent mutation in a RET enhan...   
3  A common sex-dependent mutation in a RET enhan...   
4  Studying the genetics of Hirschsprung's diseas...   

                                          chunk_text  
0  A common sex-dependent mutation in a RET enhan...  
1  The identification of common variants that con...  
2  dise

In [9]:
# character length
fixed_df["char_len"] = fixed_df["chunk_text"].str.len()

# word count
fixed_df["word_count"] = fixed_df["chunk_text"].str.split().str.len()

print("Character length distribution:")
print(fixed_df["char_len"].describe())

print("\nWord count distribution:")
print(fixed_df["word_count"].describe())

Character length distribution:
count    176626.000000
mean        370.675495
std         173.176073
min           7.000000
25%         172.000000
50%         499.000000
75%         500.000000
max         500.000000
Name: char_len, dtype: float64

Word count distribution:
count    176626.000000
mean         53.644305
std          26.168395
min           1.000000
25%          24.000000
50%          67.000000
75%          74.000000
max         119.000000
Name: word_count, dtype: float64
